# **Challenge Telecom X parte 2**

- *En este desafío, se llevará a cabo un análisis enfocado en la evasión de clientes, desarrollando un modelo predictivo para abordar este problema. Se emplearán técnicas como ingeniería de características (feature engineering) y codificación de variables (feature encoding) mediante métodos como One-Hot Encoding, entre otros. Estas transformaciones permitirán preparar los datos de manera adecuada para identificar el modelo con los parámetros que mejor se ajusten, logrando así una predicción precisa, limpia y robusta.*

---
En el análisis exploratorio anterior se realizó lo siguiente:

### 1. Extracción y Limpieza de Datos
- Importación de datos desde archivo JSON
- Normalización de estructura de datos anidados
- Tratamiento de valores nulos y duplicados
- Estandarización de variables categóricas
- Conversión de tipos de datos

### 2. Análisis Exploratorio
- Estadística descriptiva
- Visualizaciones de distribuciones
- Análisis de proporciones
- Matriz de correlación

### 3. Análisis Estadístico Inferencial
- Test de Shapiro-Wilk para verificar normalidad
- Test de Mann-Whitney U para comparación de medianas
- Test de Chi-cuadrado para asociación entre variables categóricas

## Principales Hallazgos

1. **Cargos mensuales**: Los clientes que abandonan pagan significativamente más que los clientes actuales.

2. **Antigüedad**: Los clientes nuevos tienen mayor probabilidad de abandono que los clientes con mayor tiempo en la empresa.

3. **Tipo de contrato**: Existe una fuerte relación entre el tipo de contrato y la tasa de churn:
   - Mes a mes: 42.7%
   - Anual: 11.3%
   - Bianual: 2.8%

## Tecnologías Utilizadas

- Python 3.x
- pandas
- numpy
- matplotlib
- seaborn
- scipy
- requests

## Estructura del Análisis

El notebook está organizado en las siguientes secciones:

1. Bibliotecas
2. Extracción de datos
3. Transformación de datos
4. Análisis descriptivo
5. Análisis inferencial
6. Informe final con recomendaciones

## Resultados

La tasa de churn actual es del 26.6%. El análisis identifica un perfil de cliente de alto riesgo caracterizado por ser nuevo, tener cargos mensuales elevados y contar con contrato mes a mes. Las recomendaciones se enfocan en mejorar la percepción de valor, fortalecer el onboarding de nuevos clientes e incentivar contratos de largo plazo.

## *Importando bibliotecas*

In [1]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

## *Leyendo el DataFrame*

In [2]:
data = pd.read_csv ('data_limpia.csv', sep= ',')
data.sample(5)

,customerid,churn,customer_gender,customer_seniorcitizen,customer_partner,customer_dependents,customer_tenure,phone_phoneservice,phone_multiplelines,internet_internetservice,...,internet_deviceprotection,internet_techsupport,internet_streamingtv,internet_streamingmovies,account_contract,account_paperlessbilling,account_paymentmethod,account_charges_monthly,account_charges_total,account_charges_day
1223,1813-JYWTO,0,0,0,1,0,72,1,1,Fiber optic,...,0,0,0,0,Two year,0,Bank transfer (automatic),80.45,5737.60,2.681667
4751,6701-YVNQG,0,1,0,1,0,72,1,1,DSL,...,1,1,1,1,Two year,1,Credit card (automatic),88.70,6301.70,2.956667
2963,4238-JSSWH,0,0,1,1,0,35,1,1,Fiber optic,...,1,0,1,1,Month-to-month,0,Bank transfer (automatic),102.05,3452.55,3.401667
201,0318-QUUOB,1,1,0,1,0,1,1,0,Fiber optic,...,0,0,1,0,Month-to-month,1,Electronic check,80.55,80.55,2.685000
5383,7603-USHJS,1,1,0,0,1,12,1,1,Fiber optic,...,0,0,0,0,Month-to-month,1,Credit card (automatic),73.75,871.40,2.458333


In [3]:
# ¿Modelo de regresión o clasificación?
data['churn'].nunique()

2

In [4]:
data.churn.unique()

array([0, 1])

# *Problema de clasificación*

## *Bibliotecas para modelo de clasificación*

In [16]:
# Métricas de evaluación
from yellowbrick.classifier import ConfusionMatrix, ClassificationReport, ROCAUC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Modelos de clasificación
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

In [6]:
# Onehot encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import make_column_transformer

In [7]:
data = data.drop ('customerid', axis=1)

## *Análisis exploratorio*

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   churn                      7032 non-null   int64  
 1   customer_gender            7032 non-null   int64  
 2   customer_seniorcitizen     7032 non-null   int64  
 3   customer_partner           7032 non-null   int64  
 4   customer_dependents        7032 non-null   int64  
 5   customer_tenure            7032 non-null   int64  
 6   phone_phoneservice         7032 non-null   int64  
 7   phone_multiplelines        7032 non-null   int64  
 8   internet_internetservice   7032 non-null   object 
 9   internet_onlinesecurity    7032 non-null   int64  
 10  internet_onlinebackup      7032 non-null   int64  
 11  internet_deviceprotection  7032 non-null   int64  
 12  internet_techsupport       7032 non-null   int64  
 13  internet_streamingtv       7032 non-null   int64

In [9]:
# Número de categorías por variable para saber cuántas columnas se crearán al aplicar onehot encoding
data[['internet_internetservice', 'account_contract', 'account_paymentmethod']].nunique()

internet_internetservice    3
account_contract            3
account_paymentmethod       4
dtype: int64

In [10]:
data.shape[1]

21

## *One-Hot Encoder*

*Se debe hacer una separación inicial de las X e y, dado que si no se hace al principio, el método One-Hot va a realizarse sobre la variable objetivo (y) y no es lo que se necesita.*

In [11]:
# Realizamos la separación entre variable objetivo y variables predictoras 
X = data.drop(columns='churn', axis=1)

y = data['churn']

In [14]:
one_hot = make_column_transformer ((OneHotEncoder(drop='if_binary'),
                                    X.select_dtypes(include='object').columns.to_list()),
                                    (MinMaxScaler(), X.select_dtypes(include=['float64', 'int64']).columns.to_list()),
                                remainder='passthrough',
                                sparse_threshold=0)

### *Aplicando One-Hot encoder a las variables predictoras (x)*

In [15]:
X = one_hot.fit_transform(X)
X = pd.DataFrame(X, columns=one_hot.get_feature_names_out())
display (X.sample(5))
print (X.shape)


,onehotencoder__internet_internetservice_0,onehotencoder__internet_internetservice_DSL,onehotencoder__internet_internetservice_Fiber optic,onehotencoder__account_contract_Month-to-month,onehotencoder__account_contract_One year,onehotencoder__account_contract_Two year,onehotencoder__account_paymentmethod_Bank transfer (automatic),onehotencoder__account_paymentmethod_Credit card (automatic),onehotencoder__account_paymentmethod_Electronic check,onehotencoder__account_paymentmethod_Mailed check,...,minmaxscaler__internet_onlinesecurity,minmaxscaler__internet_onlinebackup,minmaxscaler__internet_deviceprotection,minmaxscaler__internet_techsupport,minmaxscaler__internet_streamingtv,minmaxscaler__internet_streamingmovies,minmaxscaler__account_paperlessbilling,minmaxscaler__account_charges_monthly,minmaxscaler__account_charges_total,minmaxscaler__account_charges_day
865,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.917910,0.327579,0.917910
4627,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.393035,0.145863,0.393035
4179,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.015423,0.065647,0.015423
6181,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.265672,0.353860,0.265672
2256,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.634328,0.162318,0.634328


(7032, 27)
